In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error_logs = pd.read_csv("error_logs.csv")

# Parse timestamp columns to UTC datetimes (rule 10)
for df in (df_metric, df_trace, df_log, df_error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to aggregate a dataframe by cmdb_id and a name column, computing required stats
def aggregate_kpi(df, name_col, source_label):
    # Ensure 'value' is numeric
    df = df.copy()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    # Drop rows with NaN cmdb_id or name_col or value
    df = df.dropna(subset=['cmdb_id', name_col, 'value'])
    grouped = df.groupby(['cmdb_id', name_col])['value']
    agg = grouped.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p95=lambda s: s.quantile(0.95),
        p5=lambda s: s.quantile(0.05)
    ).reset_index()
    agg['source_file'] = source_label
    # Standardize KPI name column to 'kpi_name' for combined output
    agg = agg.rename(columns={name_col: 'kpi_name'})
    # Reorder columns
    agg = agg[['source_file', 'cmdb_id', 'kpi_name', 'count', 'mean', 'min', 'max', 'p95', 'p5']]
    # Cast count to int for compactness
    agg['count'] = agg['count'].astype(int)
    return agg

# Aggregate each source
agg_metric = aggregate_kpi(df_metric, 'kpi_name', 'metric.csv')
agg_trace = aggregate_kpi(df_trace, 'trace_name', 'trace.csv')
agg_log = aggregate_kpi(df_log, 'log_name', 'log.csv')

# Combine summaries
summary_df = pd.concat([agg_metric, agg_trace, agg_log], ignore_index=True)

# Limit to top 50 rows for compact display: sort for determinism
summary_df = summary_df.sort_values(['source_file', 'cmdb_id', 'kpi_name']).reset_index(drop=True)
summary_head = summary_df.head(50)

# Unique cmdb_id values across all files
unique_cmdb_ids = sorted(pd.unique(
    pd.concat([df_metric['cmdb_id'], df_trace['cmdb_id'], df_log['cmdb_id'], df_error_logs['cmdb_id']], ignore_index=True)
    .dropna()
))

# Number of distinct KPI names per component (across metric/trace/log combined)
# Build combined (cmdb_id, kpi_name) from each source, then count unique per cmdb_id
metric_kpis = df_metric[['cmdb_id', 'kpi_name']].dropna().rename(columns={'kpi_name': 'kpi_name'})
trace_kpis = df_trace[['cmdb_id', 'trace_name']].dropna().rename(columns={'trace_name': 'kpi_name'})
log_kpis = df_log[['cmdb_id', 'log_name']].dropna().rename(columns={'log_name': 'kpi_name'})
combined_kpis = pd.concat([metric_kpis, trace_kpis, log_kpis], ignore_index=True)
combined_kpis = combined_kpis.dropna(subset=['cmdb_id', 'kpi_name'])

kpi_counts = combined_kpis.drop_duplicates().groupby('cmdb_id')['kpi_name'].nunique().reset_index()
kpi_counts = kpi_counts.rename(columns={'kpi_name': 'distinct_kpi_count'}).sort_values('distinct_kpi_count', ascending=False).reset_index(drop=True)

# Compact outputs: summary_head (top 50), unique_cmdb_ids list, and kpi_counts (all components but compact)
summary_head, unique_cmdb_ids, kpi_counts

```
Out[1]:
```
Summary of what I computed
- Loaded metric.csv, trace.csv, log.csv, and error_logs.csv and parsed timestamps as UTC.
- For metric/trace/log I grouped by cmdb_id and the KPI name (kpi_name / trace_name / log_name) and computed: count, mean, min, max, global P95 and global P5 of the value series (thresholds computed from the full available series as requested).
- Returned a compact combined table (I limited the displayed rows to the top 50 for compactness).

Key compact results
- Number of unique components (cmdb_id) found across the files: 69.
- Top components by number of distinct KPI names (distinct_kpi_count — top 5):
  1. ts-preserve-service — 62 distinct KPIs
  2. ts-preserve-other-service — 62 distinct KPIs
  3. root — 60 distinct KPIs
  4. ts-travel-service — 58 distinct KPIs
  5. ts-seat-service — 54 distinct KPIs
- Bottom of the list (examples): ts-news-service has 3 distinct KPIs; several services have 4 distinct KPIs.

Observed patterns in the displayed summary (top 50 rows)
- Many log.error_count series have mean = 0.0 (no logged errors in the sampled period).
- log.row_count varies widely by service (examples from the top 50):
  - ts-order-other-service: count=25, mean≈370.8, min=177, max=490, p95≈468.0
  - ts-order-service: count=25, mean≈297.9, min=121, max=454, p95≈418.2
  - Several admin/travel services show much smaller row_count means (≈22–32).
- The compact table contains: source_file, cmdb_id, kpi_name, count, mean, min, max, p95, p5. I limited display to the first 50 rows sorted by source_file, cmdb_id, kpi_name.

If you want
- I can show more rows of the combined summary (e.g., df.head(100)), or
- drill down to a specific component to list all its KPIs and their statistics, or
- export a compact CSV-like display of the full aggregated results for further inspection. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file                      cmdb_id         kpi_name  count    mean    min    max    p95     p5
0      log.csv  ts-admin-basic-info-service  log.error_count     25    0.00    0.0    0.0    0.0    0.0
1      log.csv  ts-admin-basic-info-service    log.row_count     25   22.80   10.0   32.0   30.0   12.4
2      log.csv      ts-admin-travel-service  log.error_count     25    0.00    0.0    0.0    0.0    0.0
3      log.csv      ts-admin-travel-service    log.row_count     25   22.68    8.0   35.0   31.6   10.6
4      log.csv           ts-assurance-mongo  log.error_count     24    0.00    0.0    0.0    0.0    0.0
..         ...                          ...              ...    ...     ...    ...    ...    ...    ...
45     log.csv       ts-order-other-service    log.row_count     25  370.80  177.0  490.0  468.0  229.6
46     log.csv             ts-order-service  log.error_count     25    0.00    0.0    0.0    0.0    0.0
47     log.csv             ts-order-service    log.row_count     25  297.88  121.0  454.0  418.2  180.0
48     log.csv             ts-payment-mongo  log.error_count     24    0.00    0.0    0.0    0.0    0.0
49     log.csv             ts-payment-mongo    log.row_count     24    1.00    1.0    1.0    1.0    1.0

[50 rows x 9 columns], ['root', 'ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'],                          cmdb_id  distinct_kpi_count
0            ts-preserve-service                  62
1      ts-preserve-other-service                  62
2                           root                  60
3              ts-travel-service                  58
4                ts-seat-service                  54
..                           ...                 ...
64      ts-ticket-office-service                   4
65              ts-voucher-mysql                   4
66  ts-verification-code-service                   4
67            ts-voucher-service                   4
68               ts-news-service                   3

[69 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
from datetime import datetime, timezone

# Reuse existing loaded DataFrames if present; otherwise load them
try:
    df_metric
    df_trace
    df_log
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_trace = pd.read_csv("trace.csv")
    df_log = pd.read_csv("log.csv")
    df_error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamp columns are parsed as UTC datetimes (idempotent)
for df in (df_metric, df_trace, df_log):
    if df['timestamp'].dtype == 'int64' or df['timestamp'].dtype == 'float64' or not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Incident window (UTC)
start_ts = pd.Timestamp('2024-01-22 17:06:32', tz='UTC')
end_ts = pd.Timestamp('2024-01-22 17:36:32', tz='UTC')

# Helper: compute global thresholds DataFrame for a source
def compute_global_thresholds(df, name_col, source_label):
    tmp = df.copy()
    tmp['value'] = pd.to_numeric(tmp['value'], errors='coerce')
    tmp = tmp.dropna(subset=['cmdb_id', name_col, 'value'])
    grouped = tmp.groupby(['cmdb_id', name_col])['value']
    agg = grouped.agg(
        global_p95=lambda s: s.quantile(0.95),
        global_p5=lambda s: s.quantile(0.05)
    ).reset_index().rename(columns={name_col: 'kpi_name'})
    agg['source_file'] = source_label
    # keep ordering consistent
    agg = agg[['source_file', 'cmdb_id', 'kpi_name', 'global_p95', 'global_p5']]
    return agg

# Compute global thresholds for each source
thr_metric = compute_global_thresholds(df_metric, 'kpi_name', 'metric.csv')
thr_trace = compute_global_thresholds(df_trace, 'trace_name', 'trace.csv')
thr_log = compute_global_thresholds(df_log, 'log_name', 'log.csv')

# Filter each dataframe to candidate components and incident window
def filter_window(df, name_col):
    df2 = df.copy()
    df2['value'] = pd.to_numeric(df2['value'], errors='coerce')
    df2 = df2.dropna(subset=['cmdb_id', name_col, 'value'])
    df2 = df2[df2['cmdb_id'].isin(candidates)]
    df2 = df2[(df2['timestamp'] >= start_ts) & (df2['timestamp'] <= end_ts)]
    # unify KPI column name
    df2 = df2.rename(columns={name_col: 'kpi_name'})
    return df2[['timestamp', 'cmdb_id', 'kpi_name', 'value']]

win_metric = filter_window(df_metric, 'kpi_name')
win_trace = filter_window(df_trace, 'trace_name')
win_log = filter_window(df_log, 'log_name')

# Merge thresholds into window rows for each source and compute per-group stats
def compute_window_stats(win_df, thr_df, source_label):
    if win_df.shape[0] == 0:
        return pd.DataFrame(columns=['source_file','cmdb_id','kpi_name','global_p95','global_p5',
                                     'count_in_window','anomalies_high_count','earliest_high_timestamp',
                                     'max_value_in_window','anomalies_low_count','earliest_low_timestamp',
                                     'min_value_in_window'])
    # merge thresholds to get global p95/p5 for each row
    merged = win_df.merge(thr_df, on=['cmdb_id','kpi_name'], how='left')
    # Drop rows without thresholds (should be rare)
    merged = merged.dropna(subset=['global_p95','global_p5'])
    # Group to compute stats
    def agg_group(g):
        cnt = int(g['value'].count())
        vmax = g['value'].max()
        vmin = g['value'].min()
        gp95 = float(g['global_p95'].iloc[0])
        gp5 = float(g['global_p5'].iloc[0])
        high_mask = g['value'] >= gp95
        low_mask = g['value'] <= gp5
        high_count = int(high_mask.sum())
        low_count = int(low_mask.sum())
        # earliest timestamps if exist
        if high_count > 0:
            eh = g.loc[high_mask, 'timestamp'].min()
        else:
            eh = pd.NaT
        if low_count > 0:
            el = g.loc[low_mask, 'timestamp'].min()
        else:
            el = pd.NaT
        return pd.Series({
            'source_file': source_label,
            'cmdb_id': g['cmdb_id'].iloc[0],
            'kpi_name': g['kpi_name'].iloc[0],
            'global_p95': gp95,
            'global_p5': gp5,
            'count_in_window': cnt,
            'anomalies_high_count': high_count,
            'earliest_high_timestamp': eh,
            'max_value_in_window': vmax,
            'anomalies_low_count': low_count,
            'earliest_low_timestamp': el,
            'min_value_in_window': vmin
        })
    grouped = merged.groupby(['cmdb_id','kpi_name']).apply(agg_group).reset_index(drop=True)
    # Ensure correct dtypes
    grouped['earliest_high_timestamp'] = pd.to_datetime(grouped['earliest_high_timestamp'])
    grouped['earliest_low_timestamp'] = pd.to_datetime(grouped['earliest_low_timestamp'])
    return grouped

stats_metric = compute_window_stats(win_metric, thr_metric, 'metric.csv')
stats_trace = compute_window_stats(win_trace, thr_trace, 'trace.csv')
stats_log = compute_window_stats(win_log, thr_log, 'log.csv')

# Combine all sources
combined_stats = pd.concat([stats_metric, stats_trace, stats_log], ignore_index=True)

# Format earliest timestamps as ISO strings (UTC) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    # ensure tz-aware UTC
    if x.tzinfo is None:
        x = x.tz_localize('UTC')
    else:
        x = x.tz_convert('UTC')
    return x.isoformat()

combined_stats['earliest_high_timestamp'] = combined_stats['earliest_high_timestamp'].apply(fmt_ts)
combined_stats['earliest_low_timestamp'] = combined_stats['earliest_low_timestamp'].apply(fmt_ts)

# Sort by anomalies_high_count desc then anomalies_low_count desc
combined_stats = combined_stats.sort_values(['anomalies_high_count','anomalies_low_count'], ascending=[False, False]).reset_index(drop=True)

# Reorder columns as requested
final_cols = ['source_file','cmdb_id','kpi_name','global_p95','global_p5',
              'count_in_window','anomalies_high_count','earliest_high_timestamp',
              'max_value_in_window','anomalies_low_count','earliest_low_timestamp','min_value_in_window']
combined_stats = combined_stats[final_cols]

# Limit to top 100 rows for compact display
combined_stats_head = combined_stats.head(100)

# Per-component summary for the five candidates
# total_anomalies_in_window = sum of high+low across its KPIs
# earliest_anomaly_timestamp_across_KPIs = minimum of earliest timestamps across high/low (UTC)
# number_of_KPIs_with_any_anomaly = count of KPIs where high+low > 0

# Prepare timestamps back to datetime for computing earliest
tmp = combined_stats.copy()
tmp['eh_dt'] = pd.to_datetime(tmp['earliest_high_timestamp'], utc=True)
tmp['el_dt'] = pd.to_datetime(tmp['earliest_low_timestamp'], utc=True)
tmp['any_anomaly'] = (tmp['anomalies_high_count'] + tmp['anomalies_low_count']) > 0
summary_rows = []
for comp in candidates:
    comp_df = tmp[tmp['cmdb_id'] == comp]
    if comp_df.shape[0] == 0:
        total_anoms = 0
        earliest_anom = None
        kpis_with_any = 0
    else:
        total_anoms = int((comp_df['anomalies_high_count'] + comp_df['anomalies_low_count']).sum())
        # earliest non-null among eh_dt and el_dt
        earliest_candidates = pd.concat([comp_df['eh_dt'].dropna(), comp_df['el_dt'].dropna()])
        if earliest_candidates.empty:
            earliest_anom = None
        else:
            earliest_anom = earliest_candidates.min().isoformat()
        kpis_with_any = int(comp_df['any_anomaly'].sum())
    summary_rows.append({
        'cmdb_id': comp,
        'total_anomalies_in_window': total_anoms,
        'earliest_anomaly_timestamp_across_KPIs': earliest_anom,
        'number_of_KPIs_with_any_anomaly': kpis_with_any
    })
component_summary = pd.DataFrame(summary_rows)

# Display compact results: combined_stats_head and component_summary (5 rows)
combined_stats_head, component_summary

```
Out[2]:
```
summary = (
    "Summary of anomaly analysis for candidate services (window 2024-01-22 17:06:32 to 17:36:32 UTC):\n\n"
    "What I computed: global P95/P5 were computed from the entire series per component-KPI (no window filtering). "
    "Then I inspected telemetry points inside the incident window and counted points >= global P95 (high anomalies) "
    "and <= global P5 (low anomalies), recording earliest anomaly timestamps.\n\n"
    "Per-component highlights (sorted by total anomalies):\n"
    "- ts-travel-service: 880 total anomalies across KPIs, 58 KPIs with any anomaly. Earliest anomaly at 2024-01-22T17:09:00+00:00 UTC.\n"
    "- ts-order-service: 647 total anomalies across KPIs, 45 KPIs with any anomaly. Earliest anomaly at 2024-01-22T17:09:00+00:00 UTC.\n"
    "- ts-route-service: 442 total anomalies across KPIs, 30 KPIs with any anomaly. Earliest anomaly at 2024-01-22T17:09:00+00:00 UTC.\n"
    "- ts-train-service: 398 total anomalies across KPIs, 30 KPIs with any anomaly. Earliest anomaly at 2024-01-22T17:09:00+00:00 UTC.\n"
    "- ts-auth-service: 269 total anomalies across KPIs, 21 KPIs with any anomaly. Earliest anomaly at 2024-01-22T17:09:00+00:00 UTC.\n\n"
    "Notable pattern: several KPIs (notably 'error' and some 'error_rate' series) have global P95 and P5 equal to 0.0, "
    "so points with value 0 in the window are counted both as >=P95 and <=P5, producing high counts of both 'high' and 'low' anomalies. "
    "This inflates raw anomaly counts for those KPIs. Many earliest anomaly timestamps cluster at ~2024-01-22T17:09:00+00:00 UTC.\n\n"
    "Recommendations / next steps:\n"
    " - Inspect KPIs with non-zero global thresholds first (they give more meaningful high/low anomaly signals).\n"
    " - Drill down into specific high-impact KPIs (e.g., those with many high anomalies or with non-zero maxima inside the window).\n"
    " - Investigate events around 2024-01-22T17:09:00+00:00 UTC, as that is the earliest anomaly time seen across these services.\n\n"
    "If you want, I can: (a) list the KPIs with the largest anomaly counts for a chosen service, (b) show raw windowed time series for a KPI, "
    "or (c) re-run the anomaly counts but ignore KPIs whose global P95==P5 to avoid the zero-threshold inflation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file            cmdb_id                                           kpi_name  global_p95  global_p5  count_in_window  anomalies_high_count    earliest_high_timestamp  max_value_in_window  anomalies_low_count     earliest_low_timestamp  min_value_in_window
0   metric.csv   ts-route-service                                              error    0.000000   0.000000               25                    25  2024-01-22T17:09:00+00:00             0.000000                   25  2024-01-22T17:09:00+00:00             0.000000
1   metric.csv   ts-train-service                                              error    0.000000   0.000000               25                    25  2024-01-22T17:09:00+00:00             0.000000                   25  2024-01-22T17:09:00+00:00             0.000000
2   metric.csv  ts-travel-service                                              error    0.000000   0.000000               25                    25  2024-01-22T17:09:00+00:00             0.000000                   25  2024-01-22T17:09:00+00:00             0.000000
3    trace.csv    ts-auth-service                         trace.from_root.error_rate    0.000000   0.000000               25                    25  2024-01-22T17:09:00+00:00             0.000000                   25  2024-01-22T17:09:00+00:00             0.000000
4    trace.csv    ts-auth-service              trace.from_ts-auth-service.error_rate    0.000000   0.000000               25                    25  2024-01-22T17:09:00+00:00             0.000000                   25  2024-01-22T17:09:00+00:00             0.000000
..         ...                ...                                                ...         ...        ...              ...                   ...                        ...                  ...                  ...                        ...                  ...
95   trace.csv   ts-order-service                       trace.from_root.duration_p95    1.089204   0.875182               25                     2  2024-01-22T17:27:00+00:00             1.204479                    2  2024-01-22T17:09:00+00:00             0.865041
96   trace.csv   ts-order-service  trace.from_ts-inside-payment-service.duration_...    0.006132   0.004010               23                     2  2024-01-22T17:14:00+00:00             0.010339                    2  2024-01-22T17:09:00+00:00             0.003989
97   trace.csv   ts-order-service  trace.from_ts-inside-payment-service.duration_p95    0.014349   0.004444               23                     2  2024-01-22T17:12:00+00:00             0.030923                    2  2024-01-22T17:09:00+00:00             0.004081
98   trace.csv   ts-order-service          trace.from_ts-order-service.duration_mean    0.035271   0.012722               25                     2  2024-01-22T17:27:00+00:00             0.037976                    2  2024-01-22T17:25:00+00:00             0.007151
99   trace.csv   ts-order-service           trace.from_ts-order-service.duration_p95    0.019431   0.012503               25                     2  2024-01-22T17:20:00+00:00             0.020072                    2  2024-01-22T17:09:00+00:00             0.011498

[100 rows x 12 columns],              cmdb_id  total_anomalies_in_window earliest_anomaly_timestamp_across_KPIs  number_of_KPIs_with_any_anomaly
0    ts-auth-service                        269              2024-01-22T17:09:00+00:00                               21
1   ts-order-service                        647              2024-01-22T17:09:00+00:00                               45
2   ts-route-service                        442              2024-01-22T17:09:00+00:00                               30
3   ts-train-service                        398              2024-01-22T17:09:00+00:00                               30
4  ts-travel-service                        880              2024-01-22T17:09:00+00:00                               58)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse previously computed combined_stats and candidates
# combined_stats was produced earlier and contains the windowed stats and global thresholds.
# candidates list also exists from earlier steps.

# Ensure combined_stats is available
try:
    combined_stats
except NameError:
    raise RuntimeError("combined_stats not found in the environment. Run previous steps first.")

# Filter to the five candidate services and non-degenerate thresholds (global_p95 > global_p5)
nondeg = combined_stats[
    (combined_stats['cmdb_id'].isin(candidates)) &
    (combined_stats['global_p95'] > combined_stats['global_p5'])
].copy()

# Compute breaches_ratio per the rule:
# if global_p95 > 0: (max_value_in_window - global_p95) / global_p95 else NaN
nondeg['breaches_ratio'] = np.where(
    nondeg['global_p95'] > 0,
    (nondeg['max_value_in_window'] - nondeg['global_p95']) / nondeg['global_p95'],
    np.nan
)

# Sort by anomalies_high_count desc then breaches_ratio desc (NaNs go last)
nondeg_sorted = nondeg.sort_values(
    by=['anomalies_high_count', 'breaches_ratio'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Select and order columns as requested, limit to top 20 rows
result_table = nondeg_sorted[
    ['source_file','cmdb_id','kpi_name','global_p95','global_p5',
     'count_in_window','anomalies_high_count','earliest_high_timestamp',
     'max_value_in_window','min_value_in_window','breaches_ratio']
].head(20)

# Per-component top-3 KPIs by breaches_ratio (for the five candidates)
top3_rows = []
for comp in candidates:
    comp_df = nondeg_sorted[nondeg_sorted['cmdb_id'] == comp].copy()
    # Order by breaches_ratio desc, ignore NaNs
    comp_df = comp_df.sort_values('breaches_ratio', ascending=False, na_position='last')
    top3 = comp_df[['kpi_name','breaches_ratio']].head(3)
    # Format compactly as "kpi_name (ratio)" strings
    formatted = top3.apply(lambda r: f"{r['kpi_name']} ({np.round(r['breaches_ratio'], 4)})", axis=1).tolist()
    # Pad to 3 entries if fewer
    while len(formatted) < 3:
        formatted.append(None)
    top3_rows.append({
        'cmdb_id': comp,
        'top_kpi_1': formatted[0],
        'top_kpi_2': formatted[1],
        'top_kpi_3': formatted[2]
    })

component_top3 = pd.DataFrame(top3_rows)

# Display compact outputs: result_table (<=20 rows) and component_top3 (5 rows)
result_table, component_top3

```
Out[3]:
```
summary = (
    "Summary (non-degenerate thresholds only: global_p95 > global_p5) for the five candidate services\n\n"
    "Scope: used metric/trace/log data, kept only component-KPIs where global_p95 > global_p5, and computed breach_ratio = (max_in_window - global_p95)/global_p95 when global_p95>0.\n\n"
    "Top findings (sorted by anomalies_high_count then breach_ratio):\n"
    "- ts-order-service: the trace row_count KPIs (trace.from_root.row_count and trace.to_ts-station-service.row_count) had the highest high-anomaly counts (4 each) with breach_ratio ≈ 0.0909. "
    "Also notable: trace.from_ts-inside-payment-service.duration_p95 had the largest breach_ratio overall (≈1.155) but only 2 high anomalies.\n"
    "- ts-train-service: metric 'diskio' showed a large breach_ratio (≈0.564) with 2 high anomalies.\n"
    "- ts-travel-service: trace.from_ts-food-service.duration_p95 (breach_ratio ≈0.216) and trace.from_root.row_count (≈0.203) had measurable breaches (2 high anomalies each).\n"
    "- ts-auth-service: top breach KPIs include trace.from_root.duration_p95 (≈0.343), metric latency-90 (≈0.196), and metric diskio (≈0.156); these had small counts of high anomalies (mostly 2–3).\n"
    "- ts-route-service: top trace row_count KPIs appear with modest breach_ratios (~0.09) and a few high anomalies.\n\n"
    "Per-component top KPIs by breach_ratio (top 3 each):\n"
    "- ts-auth-service: trace.from_root.duration_p95 (0.343), latency-90 (0.1963), diskio (0.1559)\n"
    "- ts-order-service: trace.from_ts-inside-payment-service.duration_p95 (1.1550), trace.from_ts-inside-payment-service.duration_mean (0.6860), trace.from_ts-preserve-service.duration_p95 (0.5379)\n"
    "- ts-route-service: trace.from_ts-basic-service.row_count (0.0955), trace.from_ts-route-service.row_count (0.0940), trace.to_ts-route-service.row_count (0.0940)\n"
    "- ts-train-service: diskio (0.5643), cpu (0.1081), trace.from_ts-basic-service.row_count (0.0955)\n"
    "- ts-travel-service: trace.from_ts-food-service.duration_p95 (0.2165), trace.from_root.row_count (0.2030), workload (0.1094)\n\n"
    "Notes and recommended next steps:\n"
    "- Many KPIs were excluded here because their global_p95 == global_p5 (degenerate), which often occurred for series dominated by zeros; those can distort simple threshold-based counts.\n"
    "- Prioritize KPIs with both (a) non-degenerate thresholds and (b) high breach_ratio, especially when combined with multiple high-anomaly points (e.g., the inside-payment duration_p95 and train diskio).\n"
    "- Investigate events around the earliest anomaly timestamps (many cluster near 2024-01-22T17:09:00Z) and drill into traces/logs for the KPIs listed above."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file            cmdb_id                                           kpi_name     global_p95     global_p5  count_in_window  anomalies_high_count    earliest_high_timestamp  max_value_in_window  min_value_in_window  breaches_ratio
0    trace.csv   ts-order-service                          trace.from_root.row_count      11.000000      2.200000               25                     4  2024-01-22T17:12:00+00:00            12.000000             1.000000        0.090909
1    trace.csv   ts-order-service              trace.to_ts-station-service.row_count      11.000000      2.200000               25                     4  2024-01-22T17:12:00+00:00            12.000000             1.000000        0.090909
2    trace.csv   ts-train-service            trace.from_ts-travel2-service.row_count     294.000000    134.600000               25                     3  2024-01-22T17:10:00+00:00           305.000000           114.000000        0.037415
3    trace.csv    ts-auth-service                          trace.from_root.row_count      84.000000     40.400000               25                     3  2024-01-22T17:10:00+00:00            86.000000            34.000000        0.023810
4    trace.csv   ts-order-service           trace.from_ts-preserve-service.row_count       7.000000      1.000000               25                     3  2024-01-22T17:10:00+00:00             7.000000             1.000000        0.000000
5    trace.csv  ts-travel-service           trace.from_ts-preserve-service.row_count       7.000000      1.000000               24                     3  2024-01-22T17:10:00+00:00             7.000000             1.000000        0.000000
6    trace.csv   ts-order-service  trace.from_ts-inside-payment-service.duration_p95       0.014349      0.004444               23                     2  2024-01-22T17:12:00+00:00             0.030923             0.004081        1.155034
7    trace.csv   ts-order-service  trace.from_ts-inside-payment-service.duration_...       0.006132      0.004010               23                     2  2024-01-22T17:14:00+00:00             0.010339             0.003989        0.686019
8   metric.csv   ts-train-service                                             diskio  111492.473933  41975.792201               25                     2  2024-01-22T17:09:00+00:00        174408.581269         41329.348263        0.564308
9    trace.csv   ts-order-service        trace.from_ts-preserve-service.duration_p95       0.029885      0.015406               25                     2  2024-01-22T17:19:00+00:00             0.045961             0.014863        0.537936
10   trace.csv    ts-auth-service                       trace.from_root.duration_p95       0.893234      0.357152               25                     2  2024-01-22T17:14:00+00:00             1.199583             0.257569        0.342967
11   trace.csv   ts-order-service     trace.from_ts-inside-payment-service.row_count      10.600000      2.000000               23                     2  2024-01-22T17:12:00+00:00            14.000000             2.000000        0.320755
12  metric.csv   ts-order-service                                         latency-90       0.028227      0.020668               25                     2  2024-01-22T17:18:00+00:00             0.036817             0.020114        0.304299
13   trace.csv  ts-travel-service            trace.from_ts-food-service.duration_p95       0.015633      0.010954               25                     2  2024-01-22T17:26:00+00:00             0.019017             0.010855        0.216471
14   trace.csv  ts-travel-service                          trace.from_root.row_count      26.600000     14.000000               25                     2  2024-01-22T17:10:00+00:00            32.000000             7.000000        0.203008
15  metric.csv    ts-auth-service                                         latency-90       0.755629      0.390222               25                     2  2024-01-22T17:15:00+00:00             0.903990             0.380233        0.196340
16  metric.csv    ts-auth-service                                             diskio   62103.183273  44498.135763               25                     2  2024-01-22T17:09:00+00:00         71785.689972         43926.765129        0.155910
17  metric.csv    ts-auth-service                                         latency-50       0.236062      0.193308               25                     2  2024-01-22T17:13:00+00:00             0.267413             0.189341        0.132809
18   trace.csv   ts-order-service           trace.from_ts-security-service.row_count      10.600000      2.000000               25                     2  2024-01-22T17:10:00+00:00            12.000000             2.000000        0.132075
19   trace.csv   ts-order-service       trace.from_ts-preserve-service.duration_mean       0.023053      0.015060               25                     2  2024-01-22T17:16:00+00:00             0.026030             0.014863        0.129112,              cmdb_id                                          top_kpi_1                                          top_kpi_2                                          top_kpi_3
0    ts-auth-service               trace.from_root.duration_p95 (0.343)                                latency-90 (0.1963)                                    diskio (0.1559)
1   ts-order-service  trace.from_ts-inside-payment-service.duration_...  trace.from_ts-inside-payment-service.duration_...  trace.from_ts-preserve-service.duration_p95 (0...
2   ts-route-service     trace.from_ts-basic-service.row_count (0.0955)      trace.from_ts-route-service.row_count (0.094)        trace.to_ts-route-service.row_count (0.094)
3   ts-train-service                                    diskio (0.5643)                                       cpu (0.1081)     trace.from_ts-basic-service.row_count (0.0955)
4  ts-travel-service   trace.from_ts-food-service.duration_p95 (0.2165)                  trace.from_root.row_count (0.203)                                  workload (0.1094))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present
try:
    df_metric
    df_trace
    df_log
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_trace = pd.read_csv("trace.csv")
    df_log = pd.read_csv("log.csv")

# Ensure timestamps parsed as UTC datetimes
for df in (df_metric, df_trace, df_log):
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.Timestamp('2024-01-22 17:06:32', tz='UTC')
end_ts = pd.Timestamp('2024-01-22 17:36:32', tz='UTC')

# Requested KPI list per component
kpis_by_comp = {
    'ts-order-service': [
        'trace.from_ts-inside-payment-service.duration_p95',
        'trace.from_ts-inside-payment-service.duration_mean',
        'trace.from_ts-preserve-service.duration_p95',
        'trace.from_root.row_count'
    ],
    'ts-train-service': [
        'diskio',
        'trace.from_ts-travel2-service.row_count'
    ],
    'ts-auth-service': [
        'trace.from_root.duration_p95',
        'latency-90',
        'diskio'
    ],
    'ts-travel-service': [
        'trace.from_ts-food-service.duration_p95',
        'trace.from_root.row_count'
    ],
    'ts-route-service': [
        'trace.from_ts-basic-service.row_count',
        'trace.from_ts-route-service.row_count'
    ]
}

# Build unified dataframe with source and unified kpi_name column
m = df_metric.copy()
m = m.rename(columns={'kpi_name': 'kpi_name'})  # already kpi_name
m['source_file'] = 'metric.csv'
t = df_trace.copy().rename(columns={'trace_name': 'kpi_name'})
t['source_file'] = 'trace.csv'
l = df_log.copy().rename(columns={'log_name': 'kpi_name'})
l['source_file'] = 'log.csv'
df_all = pd.concat([m, t, l], ignore_index=True, sort=False)

# Ensure numeric values
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')

# Filter to only the requested component-KPI pairs
# Build set of (cmdb_id,kpi_name) to keep
pairs_to_keep = set()
for comp, klist in kpis_by_comp.items():
    for k in klist:
        pairs_to_keep.add((comp, k))

df_sel = df_all[df_all.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in pairs_to_keep, axis=1)].copy()

# Compute global thresholds (p95 and p5) from the full series (no window filtering)
thr = (
    df_sel
    .dropna(subset=['cmdb_id', 'kpi_name', 'value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95), global_p5=lambda s: s.quantile(0.05))
)

# Filter df_sel to incident window
win = df_sel[(df_sel['timestamp'] >= start_ts) & (df_sel['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed rows
win = win.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies
win['high_anomaly'] = win['value'] >= win['global_p95']
win['low_anomaly'] = win['value'] <= win['global_p5']

# Prepare raw windowed rows for output (limit later)
raw_cols = ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'high_anomaly', 'low_anomaly']
raw_windowed = win[raw_cols].sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Identify consecutive high_anomaly segments per (cmdb_id, kpi_name)
segments = []
grp = win.sort_values(['cmdb_id','kpi_name','timestamp']).groupby(['cmdb_id','kpi_name'], sort=False)

for (cmdb_id, kpi_name), g in grp:
    g = g.reset_index(drop=True)
    # ensure sorted by timestamp
    g = g.sort_values('timestamp').reset_index(drop=True)
    # high mask
    high = g['high_anomaly'].fillna(False).astype(bool)
    if not high.any():
        continue  # no high segments
    # identify contiguous segments: change points where high differs from previous
    segment_id = (high != high.shift(fill_value=False)).cumsum()
    g['segment_id'] = segment_id
    # select only segments where high==True
    high_segments = g[high].groupby('segment_id', sort=False)
    for seg_id, sg in high_segments:
        fault_start = sg['timestamp'].min()
        fault_end = sg['timestamp'].max()
        points_in_fault = int(sg.shape[0])
        duration_minutes = points_in_fault  # one point per minute
        max_value_in_fault = float(sg['value'].max())
        # lookup global_p95 for this (cmdb_id,kpi_name)
        row_thr = thr[(thr['cmdb_id']==cmdb_id) & (thr['kpi_name']==kpi_name)]
        if row_thr.shape[0] == 0:
            gp95 = np.nan
        else:
            gp95 = float(row_thr['global_p95'].iloc[0])
        if gp95 > 0 and not np.isnan(gp95):
            breach_ratio = (max_value_in_fault - gp95) / gp95
        else:
            breach_ratio = np.nan
        segments.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'fault_start_timestamp': fault_start,
            'fault_end_timestamp': fault_end,
            'duration_minutes': duration_minutes,
            'points_in_fault': points_in_fault,
            'max_value_in_fault': max_value_in_fault,
            'global_p95': gp95,
            'breach_ratio': breach_ratio
        })

segments_df = pd.DataFrame(segments)

# Sort segments as requested: duration_minutes desc then breach_ratio desc
if not segments_df.empty:
    segments_df = segments_df.sort_values(['duration_minutes','breach_ratio'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20 rows
segments_head = segments_df.head(20)

# Prepare raw windowed rows limited to first 50 rows (sorted for context)
raw_head = raw_windowed.head(50)

# Ensure timestamps are shown as ISO strings for compactness in outputs
# but keep underlying dtype; create display columns
segments_head_disp = segments_head.copy()
if not segments_head_disp.empty:
    segments_head_disp['fault_start_timestamp'] = segments_head_disp['fault_start_timestamp'].dt.tz_convert('UTC').astype(str)
    segments_head_disp['fault_end_timestamp'] = segments_head_disp['fault_end_timestamp'].dt.tz_convert('UTC').astype(str)

raw_head_disp = raw_head.copy()
if not raw_head_disp.empty:
    raw_head_disp['timestamp'] = raw_head_disp['timestamp'].dt.tz_convert('UTC').astype(str)

# Final compact outputs
segments_head_disp, raw_head_disp

```
Out[4]:
```
summary = (
    "Summary of fault-segment analysis for the requested KPIs (window 2024-01-22 17:06:32 → 17:36:32 UTC):\n\n"
    "What I did: For each listed KPI I computed global P95/P5 from the full series, filtered rows to the incident window, "
    "marked high/low anomalies, and grouped consecutive high_anomaly points into fault segments. I report segments and up to 50 raw windowed rows for context.\n\n"
    "Key findings (most important fault segments):\n"
    "- ts-order-service (inside-payment traces): a short multi-point fault from 2024-01-22T17:14:00Z to 17:15:00Z (2 points) "
    "on trace.from_ts-inside-payment-service.duration_* with max value 0.010339, global P95=0.006132, breach_ratio ≈ 0.686. "
    "Also a single-point spike in duration_p95 at 17:15 with breach_ratio ≈ 1.155 (largest relative breach observed).\n"
    "- ts-train-service (diskio): a 2-minute segment at 2024-01-22T17:09:00Z–17:10:00Z with max diskio ≈ 174,408, global P95 ≈ 111,492 and breach_ratio ≈ 0.564. "
    "This is the largest absolute breach on a metric in the set.\n"
    "- ts-order-service (trace.from_root.row_count): two short segments (one 2-minute at 17:12–17:13 and one 1-minute at 17:18) with small relative breaches (breach_ratio ≈ 0–0.091).\n"
    "- ts-auth-service: single-point high anomalies in trace.from_root.duration_p95 (17:14, breach_ratio ≈ 0.343) and metric latency-90 / diskio with modest breach ratios (~0.02–0.20).\n"
    "- ts-travel-service: single-point anomalies in trace.from_ts-food-service.duration_p95 (17:26, breach_ratio ≈ 0.216) and trace.from_root.row_count (17:10, breach_ratio ≈ 0.203).\n"
    "- ts-route-service: single-point row_count anomalies at 17:10 with modest breach_ratios (~0.094–0.096).\n\n"
    "Interpretation and priorities:\n"
    "- Highest-priority items: the inside-payment trace spikes (order service) and the train diskio spike — they show the largest relative/absolute breaches and multi-point segments.\n"
    "- Many other anomalies are single-minute spikes or small relative breaches; they are lower priority but worth inspecting if correlated across services.\n"
    "- Some KPIs had degenerate thresholds and were excluded from this summary; focusing on non-degenerate KPIs reduces false positives.\n\n"
    "Context rows: I also returned up to 50 raw windowed rows (timestamp, cmdb_id, kpi_name, value, high_anomaly, low_anomaly) to help inspect timing and values.\n\n"
    "Next steps (suggested):\n"
    "1) Drill into traces/logs for the inside-payment and train.diskio spikes (check call stacks, dependent services, and resource metrics).\n"
    "2) Correlate these fault timestamps with other system events (logs, infrastructure metrics) around 17:09–17:15 UTC.\n"
    "3) If needed, expand the window slightly to see pre/post behavior and confirm impact duration.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name      fault_start_timestamp        fault_end_timestamp  duration_minutes  points_in_fault  max_value_in_fault     global_p95  breach_ratio
0    ts-order-service  trace.from_ts-inside-payment-service.duration_...  2024-01-22 17:14:00+00:00  2024-01-22 17:15:00+00:00                 2                2            0.010339       0.006132      0.686019
1    ts-train-service                                             diskio  2024-01-22 17:09:00+00:00  2024-01-22 17:10:00+00:00                 2                2       174408.581269  111492.473933      0.564308
2    ts-order-service                          trace.from_root.row_count  2024-01-22 17:12:00+00:00  2024-01-22 17:13:00+00:00                 2                2           11.000000      11.000000      0.000000
3    ts-order-service  trace.from_ts-inside-payment-service.duration_p95  2024-01-22 17:15:00+00:00  2024-01-22 17:15:00+00:00                 1                1            0.030923       0.014349      1.155034
4    ts-order-service        trace.from_ts-preserve-service.duration_p95  2024-01-22 17:19:00+00:00  2024-01-22 17:19:00+00:00                 1                1            0.045961       0.029885      0.537936
5     ts-auth-service                       trace.from_root.duration_p95  2024-01-22 17:14:00+00:00  2024-01-22 17:14:00+00:00                 1                1            1.199583       0.893234      0.342967
6   ts-travel-service            trace.from_ts-food-service.duration_p95  2024-01-22 17:33:00+00:00  2024-01-22 17:33:00+00:00                 1                1            0.019017       0.015633      0.216471
7   ts-travel-service                          trace.from_root.row_count  2024-01-22 17:10:00+00:00  2024-01-22 17:10:00+00:00                 1                1           32.000000      26.600000      0.203008
8     ts-auth-service                                         latency-90  2024-01-22 17:17:00+00:00  2024-01-22 17:17:00+00:00                 1                1            0.903990       0.755629      0.196340
9     ts-auth-service                                             diskio  2024-01-22 17:09:00+00:00  2024-01-22 17:09:00+00:00                 1                1        71785.689972   62103.183273      0.155910
10   ts-route-service              trace.from_ts-basic-service.row_count  2024-01-22 17:10:00+00:00  2024-01-22 17:10:00+00:00                 1                1          156.000000     142.400000      0.095506
11   ts-route-service              trace.from_ts-route-service.row_count  2024-01-22 17:10:00+00:00  2024-01-22 17:10:00+00:00                 1                1         2282.000000    2086.000000      0.093960
12   ts-order-service                          trace.from_root.row_count  2024-01-22 17:18:00+00:00  2024-01-22 17:18:00+00:00                 1                1           12.000000      11.000000      0.090909
13   ts-order-service  trace.from_ts-inside-payment-service.duration_p95  2024-01-22 17:12:00+00:00  2024-01-22 17:12:00+00:00                 1                1            0.014914       0.014349      0.039320
14   ts-train-service            trace.from_ts-travel2-service.row_count  2024-01-22 17:10:00+00:00  2024-01-22 17:10:00+00:00                 1                1          305.000000     294.000000      0.037415
15   ts-order-service        trace.from_ts-preserve-service.duration_p95  2024-01-22 17:24:00+00:00  2024-01-22 17:24:00+00:00                 1                1            0.030627       0.029885      0.024832
16    ts-auth-service                       trace.from_root.duration_p95  2024-01-22 17:17:00+00:00  2024-01-22 17:17:00+00:00                 1                1            0.914356       0.893234      0.023646
17  ts-travel-service                          trace.from_root.row_count  2024-01-22 17:18:00+00:00  2024-01-22 17:18:00+00:00                 1                1           27.000000      26.600000      0.015038
18  ts-travel-service            trace.from_ts-food-service.duration_p95  2024-01-22 17:26:00+00:00  2024-01-22 17:26:00+00:00                 1                1            0.015777       0.015633      0.009227
19    ts-auth-service                                             diskio  2024-01-22 17:30:00+00:00  2024-01-22 17:30:00+00:00                 1                1        62380.754661   62103.183273      0.004470,                     timestamp          cmdb_id    kpi_name         value  high_anomaly  low_anomaly
0   2024-01-22 17:09:00+00:00  ts-auth-service      diskio  71785.689972          True        False
1   2024-01-22 17:10:00+00:00  ts-auth-service      diskio  45699.736067         False        False
2   2024-01-22 17:11:00+00:00  ts-auth-service      diskio  52030.982787         False        False
3   2024-01-22 17:12:00+00:00  ts-auth-service      diskio  56293.064615         False        False
4   2024-01-22 17:13:00+00:00  ts-auth-service      diskio  55032.965056         False        False
..                        ...              ...         ...           ...           ...          ...
45  2024-01-22 17:29:00+00:00  ts-auth-service  latency-90      0.405856         False        False
46  2024-01-22 17:30:00+00:00  ts-auth-service  latency-90      0.386313         False         True
47  2024-01-22 17:31:00+00:00  ts-auth-service  latency-90      0.446856         False        False
48  2024-01-22 17:32:00+00:00  ts-auth-service  latency-90      0.445030         False        False
49  2024-01-22 17:33:00+00:00  ts-auth-service  latency-90      0.455207         False        False

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import re

# Load trace.csv if not already loaded
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Parse timestamp to UTC datetime if needed
if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure numeric values
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Incident window (UTC)
start_ts = pd.Timestamp('2024-01-22 17:06:32', tz='UTC')
end_ts = pd.Timestamp('2024-01-22 17:36:32', tz='UTC')

# 1) Compute global_p95 per (cmdb_id, trace_name) from full trace series
thr_trace_full = (
    df_trace
    .dropna(subset=['cmdb_id', 'trace_name', 'value'])
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95))
)

# 2) Filter trace rows to incident window
trace_win = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge global_p95 into windowed rows
trace_win = trace_win.merge(thr_trace_full, on=['cmdb_id', 'trace_name'], how='left')

# 3) Mark high_anomaly if value >= global_p95 (handle missing thresholds as False)
trace_win['high_anomaly'] = False
mask_valid = trace_win['global_p95'].notna()
trace_win.loc[mask_valid, 'high_anomaly'] = trace_win.loc[mask_valid, 'value'] >= trace_win.loc[mask_valid, 'global_p95']

# Parse trace_name to extract direction and peer_service
trace_win['direction'] = 'other'
trace_win.loc[trace_win['trace_name'].str.contains(r'\bfrom_', na=False), 'direction'] = 'from'
trace_win.loc[trace_win['trace_name'].str.contains(r'\bto_', na=False), 'direction'] = 'to'

# Extract peer_service after from_ or to_, stopping at next dot or end
trace_win['peer_service'] = trace_win['trace_name'].str.extract(r'\bfrom_([^.\s]+)', expand=False)
to_peers = trace_win['trace_name'].str.extract(r'\bto_([^.\s]+)', expand=False)
trace_win['peer_service'] = trace_win['peer_service'].fillna(to_peers)
# Keep None for missing peers
trace_win['peer_service'] = trace_win['peer_service'].where(trace_win['peer_service'].notna(), None)

# 4) Aggregate per cmdb_id, direction, peer_service, trace_name
# Base aggregations: count, sum(high_anomaly) as anomalies_high_count, max value
agg_base = (
    trace_win
    .groupby(['cmdb_id', 'direction', 'peer_service', 'trace_name'], dropna=False, as_index=False)
    .agg(
        count_in_window=('value', 'count'),
        anomalies_high_count=('high_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# Compute earliest_high_timestamp per group from rows where high_anomaly==True
high_rows = trace_win[trace_win['high_anomaly']].copy()
if not high_rows.empty:
    earliest_high = (
        high_rows
        .groupby(['cmdb_id', 'direction', 'peer_service', 'trace_name'], dropna=False, as_index=False)
        .agg(earliest_high_timestamp=('timestamp', 'min'))
    )
else:
    # empty DataFrame with expected columns
    earliest_high = pd.DataFrame(columns=['cmdb_id','direction','peer_service','trace_name','earliest_high_timestamp'])

# Merge earliest_high into agg_base
agg = agg_base.merge(earliest_high, on=['cmdb_id','direction','peer_service','trace_name'], how='left')

# Merge global_p95 from thr_trace_full (on cmdb_id, trace_name)
agg = agg.merge(thr_trace_full, on=['cmdb_id','trace_name'], how='left')

# Keep only rows where anomalies_high_count > 0
agg_pos = agg[agg['anomalies_high_count'] > 0].copy()

# Format earliest_high_timestamp as ISO strings (UTC) for compact display
agg_pos['earliest_high_timestamp'] = pd.to_datetime(agg_pos['earliest_high_timestamp'], utc=True)
agg_pos['earliest_high_timestamp'] = agg_pos['earliest_high_timestamp'].dt.tz_convert('UTC').astype(str)

# Select and order columns as requested
main_table = agg_pos[[
    'cmdb_id','direction','peer_service','trace_name',
    'count_in_window','anomalies_high_count','earliest_high_timestamp',
    'max_value_in_window','global_p95'
]].copy()

# Sort by anomalies_high_count desc then max_value_in_window desc, limit to top 50
main_table = main_table.sort_values(['anomalies_high_count','max_value_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

# 5) Per-candidate compact summary for the five services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
summary_rows = []
for comp in candidates:
    comp_rows = trace_win[trace_win['cmdb_id'] == comp].copy()
    total_trace_anomalies = int(comp_rows['high_anomaly'].sum())
    if total_trace_anomalies > 0:
        earliest_ts = comp_rows.loc[comp_rows['high_anomaly'], 'timestamp'].min()
        earliest_ts_str = pd.to_datetime(earliest_ts, utc=True).tz_convert('UTC').isoformat()
    else:
        earliest_ts_str = None
    # Peer services with anomalies: count high_anomaly per peer_service
    peers = (
        comp_rows[comp_rows['high_anomaly']]
        .groupby('peer_service', dropna=False)['high_anomaly']
        .sum()
        .reset_index()
        .rename(columns={'high_anomaly':'anomaly_count'})
    )
    if not peers.empty:
        peers = peers.sort_values('anomaly_count', ascending=False)
        # format top 5 peers as "peer (count)"; replace None with 'other'
        peers['peer_label'] = peers['peer_service'].fillna('other').astype(str) + ' (' + peers['anomaly_count'].astype(int).astype(str) + ')'
        top_peers_list = peers['peer_label'].head(5).tolist()
    else:
        top_peers_list = []
    peers_str = ', '.join(top_peers_list) if top_peers_list else None
    summary_rows.append({
        'cmdb_id': comp,
        'total_trace_anomalies_in_window': total_trace_anomalies,
        'earliest_trace_anomaly_timestamp': earliest_ts_str,
        'list_of_peer_services_with_anomalies': peers_str
    })

candidate_summary = pd.DataFrame(summary_rows)

# Final compact outputs
main_table, candidate_summary

```
Out[5]:
```
summary = (
    "Trace anomaly summary (window 2024-01-22 17:06:32 → 17:36:32 UTC):\n\n"
    "Per-candidate overview:\n"
)

# Use existing candidate_summary and main_table from the environment if available
try:
    cs = candidate_summary.copy()
    mt = main_table.copy()
except NameError:
    summary += "No previously computed results found in the environment.\n"
else:
    for _, r in cs.iterrows():
        peers = r['list_of_peer_services_with_anomalies'] or "none"
        earliest = r['earliest_trace_anomaly_timestamp'] or "none"
        summary += f"- {r['cmdb_id']}: total_trace_anomalies_in_window={r['total_trace_anomalies_in_window']}, earliest={earliest}, top_peers={peers}\n"

    # Note degenerate/global-zero thresholds
    degenerate_rows = mt[(mt['global_p95'] == 0)].shape[0]
    total_rows = mt.shape[0]
    summary += (
        "\nObservations:\n"
        f"- {degenerate_rows} of {total_rows} aggregated trace rows in the main table have global_p95 == 0; "
        "many trace.error_rate series are zero-valued, which makes global_p95==0 and causes all window points to be marked as 'high' by the >=P95 rule.\n"
        "- Earliest high-anomaly timestamps are heavily clustered at 2024-01-22T17:09:00+00:00 across many traces.\n\n"
        "Recommendations:\n"
        "- Focus first on non-degenerate anomaly groups (global_p95 > 0) and on peers with multiple anomalies.\n"
        "- Investigate the time around 2024-01-22T17:09:00+00:00 for correlated events across services.\n"
        "- If needed, re-evaluate anomaly logic for zero-dominated error_rate series (e.g., use different thresholds or require value > P95).\n"
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id direction                 peer_service                                       trace_name  count_in_window  anomalies_high_count    earliest_high_timestamp  max_value_in_window  global_p95
0                  root        to  ts-admin-basic-info-service  trace.to_ts-admin-basic-info-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
1                  root        to      ts-admin-travel-service      trace.to_ts-admin-travel-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
2                  root        to         ts-assurance-service         trace.to_ts-assurance-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
3                  root        to              ts-auth-service              trace.to_ts-auth-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
4                  root        to           ts-consign-service           trace.to_ts-consign-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
..                  ...       ...                          ...                                              ...              ...                   ...                        ...                  ...         ...
45  ts-contacts-service        to          ts-contacts-service          trace.to_ts-contacts-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
46  ts-food-map-service      from          ts-food-map-service        trace.from_ts-food-map-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
47  ts-food-map-service      from              ts-food-service            trace.from_ts-food-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
48  ts-food-map-service        to          ts-food-map-service          trace.to_ts-food-map-service.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0
49      ts-food-service      from                         root                       trace.from_root.error_rate               25                    25  2024-01-22 17:09:00+00:00                  0.0         0.0

[50 rows x 9 columns],              cmdb_id  total_trace_anomalies_in_window earliest_trace_anomaly_timestamp               list_of_peer_services_with_anomalies
0    ts-auth-service                               94        2024-01-22T17:09:00+00:00                    ts-auth-service (62), root (32)
1   ts-order-service                              282        2024-01-22T17:09:00+00:00  ts-order-service (62), root (33), ts-station-s...
2   ts-route-service                              155        2024-01-22T17:09:00+00:00  ts-route-service (62), ts-basic-service (31), ...
3   ts-train-service                              156        2024-01-22T17:09:00+00:00  ts-train-service (62), ts-travel2-service (32)...
4  ts-travel-service                              372        2024-01-22T17:09:00+00:00  ts-seat-service (62), ts-travel-service (62), ...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if available, otherwise load
try:
    df_log
    df_error_logs
except NameError:
    df_log = pd.read_csv("log.csv")
    df_error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamps are parsed as UTC datetimes
for df in (df_log, df_error_logs):
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Candidate components and incident window (UTC)
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
start_ts = pd.Timestamp('2024-01-22 17:06:32', tz='UTC')
end_ts = pd.Timestamp('2024-01-22 17:36:32', tz='UTC')

# 1) Process log.csv: filter to window and components
log_win = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(candidates))
].copy()

# Ensure numeric 'value'
log_win['value'] = pd.to_numeric(log_win['value'], errors='coerce')

# Aggregate per cmdb_id and log_name using windowed rows
logs_agg = (
    log_win
    .dropna(subset=['cmdb_id','log_name'])
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(
        count_in_window='count',
        mean_value='mean',
        max_value='max',
        p95_value=lambda s: s.quantile(0.95)
    )
)

# Flag rows where log_name is 'log.error_count' and max_value > 0
logs_agg['error_count_flag'] = np.where(
    (logs_agg['log_name'] == 'log.error_count') & (logs_agg['max_value'] > 0),
    True,
    False
)

# Sort by max_value desc and keep top 50 for compactness
logs_agg = logs_agg.sort_values('max_value', ascending=False).reset_index(drop=True)
logs_agg_head = logs_agg.head(50)

# 2) Process error_logs.csv: filter to window and components, return up to 20 raw rows
err_win = df_error_logs[
    (df_error_logs['timestamp'] >= start_ts) &
    (df_error_logs['timestamp'] <= end_ts) &
    (df_error_logs['cmdb_id'].isin(candidates))
].copy()

# Sort by timestamp ascending and select columns; format timestamp as ISO UTC string for compact display
err_win = err_win.sort_values('timestamp', ascending=True).reset_index(drop=True)
err_win_head = err_win[['timestamp','cmdb_id','message']].head(20).copy()
err_win_head['timestamp'] = err_win_head['timestamp'].dt.tz_convert('UTC').astype(str)

# Display compact outputs
logs_agg_head, err_win_head

```
Out[6]:
```
summary = (
    "Summary of logs and error messages for the five candidate services (window 2024-01-22 17:06:32 → 17:36:32 UTC):\n\n"
    "Log aggregates (log.csv):\n"
    "- ts-travel-service (log.row_count): count=25, mean=1404.08, max=2053, p95≈1969.2\n"
    "- ts-route-service  (log.row_count): count=25, mean=841.28,  max=1141, p95≈1043.0\n"
    "- ts-order-service  (log.row_count): count=25, mean=297.88,  max=454,  p95≈418.2\n"
    "- ts-auth-service   (log.row_count): count=25, mean=273.44,  max=342,  p95≈337.6\n"
    "- ts-train-service  (log.row_count): count=3,  mean≈1.67,    max=2,    p95=2.0\n\n"
    "Log error counts (log.error_count):\n"
    "- For all five services, log.error_count had mean=0 and max=0 in the window (no flagged error_count > 0).\n\n"
    "Error logs (error_logs.csv):\n"
    "- No error_log rows were found for these components in the incident window (0 messages returned).\n\n"
    "Notes:\n"
    "- The largest volumes are in ts-travel-service and ts-route-service (high row_count). ts-train-service has very sparse log rows in this window (only 3 samples).\n"
    "- There were no logged error counts or raw error messages for these services in this window, so investigate other telemetry (metrics/traces) or extend the time window if you expect related events.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count_in_window   mean_value  max_value  p95_value  error_count_flag
0  ts-travel-service    log.row_count               25  1404.080000       2053     1969.2             False
1   ts-route-service    log.row_count               25   841.280000       1141     1043.0             False
2   ts-order-service    log.row_count               25   297.880000        454      418.2             False
3    ts-auth-service    log.row_count               25   273.440000        342      337.6             False
4   ts-train-service    log.row_count                3     1.666667          2        2.0             False
5    ts-auth-service  log.error_count               25     0.000000          0        0.0             False
6   ts-route-service  log.error_count               25     0.000000          0        0.0             False
7   ts-order-service  log.error_count               25     0.000000          0        0.0             False
8   ts-train-service  log.error_count                3     0.000000          0        0.0             False
9  ts-travel-service  log.error_count               25     0.000000          0        0.0             False, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetime (idempotent)
if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure numeric values
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Incident window (UTC)
start_ts = pd.Timestamp('2024-01-22 17:06:32', tz='UTC')
end_ts = pd.Timestamp('2024-01-22 17:36:32', tz='UTC')

# Candidate services
candidates = {'ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service'}

# 1) Compute global_p95 per (cmdb_id, trace_name) using full trace series
thr_trace_full = (
    df_trace
    .dropna(subset=['cmdb_id','trace_name','value'])
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95))
)

# 2) Filter trace rows to incident window
trace_win_all = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Parse trace_name to extract direction and peer_service
trace_win_all['direction'] = 'other'
trace_win_all.loc[trace_win_all['trace_name'].str.contains(r'\bfrom_', na=False), 'direction'] = 'from'
trace_win_all.loc[trace_win_all['trace_name'].str.contains(r'\bto_', na=False), 'direction'] = 'to'

# extract peer after from_ or to_, stop at next dot or end
trace_win_all['peer_service'] = trace_win_all['trace_name'].str.extract(r'\bfrom_([^.\s]+)', expand=False)
to_peers = trace_win_all['trace_name'].str.extract(r'\bto_([^.\s]+)', expand=False)
trace_win_all['peer_service'] = trace_win_all['peer_service'].fillna(to_peers)
trace_win_all['peer_service'] = trace_win_all['peer_service'].where(trace_win_all['peer_service'].notna(), None)

# 4) Restrict rows to those where peer_service is in candidates OR cmdb_id is in candidates
mask_peer = trace_win_all['peer_service'].isin(candidates)
mask_cmdb = trace_win_all['cmdb_id'].isin(candidates)
trace_win = trace_win_all[mask_peer | mask_cmdb].copy()

# Merge global_p95 into windowed rows
trace_win = trace_win.merge(thr_trace_full, on=['cmdb_id','trace_name'], how='left')

# Mark high_anomaly if value >= global_p95 (require non-null global_p95)
trace_win['high_anomaly'] = False
valid_thr = trace_win['global_p95'].notna()
trace_win.loc[valid_thr, 'high_anomaly'] = trace_win.loc[valid_thr, 'value'] >= trace_win.loc[valid_thr, 'global_p95']

# Aggregate by cmdb_id, direction, peer_service
agg = (
    trace_win
    .groupby(['cmdb_id','direction','peer_service'], dropna=False, as_index=False)
    .agg(
        count_in_window=('value','count'),
        anomalies_high_count=('high_anomaly','sum'),
        max_value_in_window=('value','max'),
        # choose a representative global_p95 for the group: use max across trace_names in the group
        global_p95=('global_p95', lambda s: np.nan if s.dropna().empty else s.dropna().max())
    )
)

# Compute earliest_high_timestamp per group (from rows where high_anomaly==True)
high_rows = trace_win[trace_win['high_anomaly']].copy()
if not high_rows.empty:
    earliest = (
        high_rows
        .groupby(['cmdb_id','direction','peer_service'], dropna=False, as_index=False)
        .agg(earliest_high_timestamp=('timestamp','min'))
    )
    agg = agg.merge(earliest, on=['cmdb_id','direction','peer_service'], how='left')
else:
    agg['earliest_high_timestamp'] = pd.NaT

# Keep only groups with anomalies_high_count > 0
agg_pos = agg[agg['anomalies_high_count'] > 0].copy()

# Format earliest_high_timestamp as ISO string (UTC) for compact display
agg_pos['earliest_high_timestamp'] = pd.to_datetime(agg_pos['earliest_high_timestamp'], utc=True)
agg_pos['earliest_high_timestamp'] = agg_pos['earliest_high_timestamp'].dt.tz_convert('UTC').astype(str)

# Sort by anomalies_high_count desc then max_value_in_window desc, limit to top 50
agg_table = agg_pos.sort_values(['anomalies_high_count','max_value_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

# Prepare raw trace rows referencing these services (either as cmdb_id or peer), include requested columns
raw_cols = ['timestamp','cmdb_id','trace_name','value','direction','peer_service','high_anomaly']
raw_rows = trace_win[raw_cols].sort_values(['timestamp','cmdb_id']).head(50).copy()
# Format timestamp as ISO UTC string for compact display
raw_rows['timestamp'] = raw_rows['timestamp'].dt.tz_convert('UTC').astype(str)

# Display compact aggregated and raw tables
agg_table, raw_rows

```
Out[7]:
```
Summary of trace findings for the five services (window 2024-01-22 17:06:32 → 17:36:32 UTC)

What I observed
- Many trace groups that involve the five services show high-anomaly counts concentrated around ~2024-01-22T17:09:00Z. Anomalies are clustered across call chains (root ↔ services and service→service traces).
- Example aggregated observations from the returned table:
  - root → ts-order-service (trace.to_ts-order-service.*): count_in_window=100, anomalies_high_count=33, global_p95=11.0, max_value_in_window=12.0, earliest_high_timestamp=2024-01-22T17:09:00Z.
  - ts-order-service ← root (trace.from_root.*): same counts (count_in_window=100, anomalies_high_count=33, global_p95=11.0, max=12.0).
  - ts-order-service → ts-station-service: count_in_window=100, anomalies_high_count=33, global_p95=11.0, max=12.0 (earliest 17:09).
  - ts-train-service ← ts-travel2-service: count_in_window=100, anomalies_high_count=32, global_p95=294.0, max_value_in_window=305.0 (earliest 17:09).
  - ts-travel-service and ts-preserve/ts-seat related groups show ~31 high anomalies each (earliest 17:09) with small numeric gaps between max and global_p95 (e.g., max 11 vs p95 10.8).
- The raw sample rows show many trace.error_rate entries flagged high at 17:09:00 (value 0.0). These appear throughout call chains (root → ts-auth-service, root → ts-order-service, ts-order-service.from_root.error_rate, etc.).

Important caveat
- A large number of “high_anomaly” flags come from trace.error_rate series where global_p95 == 0.0. Because the rule used was value >= global_p95, zero-valued series with global_p95==0 mark all zero points as high. This artificially inflates anomaly counts for many groups. In other words, many flagged “high” points are due to degenerate zero thresholds rather than meaningful positive spikes.

Practical guidance / next steps
- Prioritize groups with global_p95 > 0 for meaningful anomalies (examples: ts-train-service ← ts-travel2_service row_count, some ts-order-service row_count traces). These show real numeric breaches (e.g., train row_count max 305 vs p95 294).
- Investigate the time window around 2024-01-22T17:09:00Z for correlated events across services (traces, metrics, logs).
- For error_rate traces consider changing the rule (e.g., require value > global_p95) or exclude zero-dominated series when counting anomalies to avoid false positives.
- Use the provided raw trace rows to follow call chains (timestamps and peer_service) for deeper debugging of specific interactions (order ↔ station, train ↔ travel2, etc.).

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id direction             peer_service  count_in_window  anomalies_high_count  max_value_in_window  global_p95    earliest_high_timestamp
0                      root        to         ts-order-service              100                    33                 12.0        11.0  2024-01-22 17:09:00+00:00
1          ts-order-service      from                     root              100                    33                 12.0        11.0  2024-01-22 17:09:00+00:00
2          ts-order-service        to       ts-station-service              100                    33                 12.0        11.0  2024-01-22 17:09:00+00:00
3        ts-station-service      from         ts-order-service              100                    33                 12.0        11.0  2024-01-22 17:09:00+00:00
4          ts-train-service      from       ts-travel2-service              100                    32                305.0       294.0  2024-01-22 17:09:00+00:00
..                      ...       ...                      ...              ...                   ...                  ...         ...                        ...
45      ts-security-service        to         ts-order-service              100                    31                 12.0        10.6  2024-01-22 17:09:00+00:00
46  ts-admin-travel-service        to        ts-travel-service              100                    31                 11.0        10.8  2024-01-22 17:09:00+00:00
47        ts-travel-service      from  ts-admin-travel-service              100                    31                 11.0        10.8  2024-01-22 17:09:00+00:00
48      ts-preserve-service        to        ts-travel-service               96                    31                  7.0         7.0  2024-01-22 17:09:00+00:00
49        ts-travel-service      from      ts-preserve-service               96                    31                  7.0         7.0  2024-01-22 17:09:00+00:00

[50 rows x 8 columns],                     timestamp           cmdb_id                                         trace_name      value direction               peer_service  high_anomaly
0   2024-01-22 17:09:00+00:00              root             trace.to_ts-auth-service.duration_mean   0.225064        to            ts-auth-service         False
1   2024-01-22 17:09:00+00:00              root              trace.to_ts-auth-service.duration_p95   0.366225        to            ts-auth-service         False
2   2024-01-22 17:09:00+00:00              root                trace.to_ts-auth-service.error_rate   0.000000        to            ts-auth-service          True
3   2024-01-22 17:09:00+00:00              root                 trace.to_ts-auth-service.row_count  36.000000        to            ts-auth-service         False
4   2024-01-22 17:09:00+00:00              root            trace.to_ts-order-service.duration_mean   0.847955        to           ts-order-service         False
..                        ...               ...                                                ...        ...       ...                        ...           ...
45  2024-01-22 17:09:00+00:00  ts-order-service                       trace.from_root.duration_p95   0.865041      from                       root         False
46  2024-01-22 17:09:00+00:00  ts-order-service                         trace.from_root.error_rate   0.000000      from                       root          True
47  2024-01-22 17:09:00+00:00  ts-order-service                          trace.from_root.row_count   3.000000      from                       root         False
48  2024-01-22 17:09:00+00:00  ts-order-service  trace.from_ts-inside-payment-service.duration_...   0.003989      from  ts-inside-payment-service         False
49  2024-01-22 17:09:00+00:00  ts-order-service  trace.from_ts-inside-payment-service.duration_p95   0.004381      from  ts-inside-payment-service         False

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```